# ESC-50: Explore Environmental Sounds with Audio Embeddings

This notebook loads the [ESC-50](https://github.com/karolpiczak/ESC-50) environmental sound classification dataset from Hugging Face, creates a 3LC Table with playable audio samples, extracts embeddings using a pretrained Audio Spectrogram Transformer (AST), and reduces them to 2D with PaCMAP.

The result is an interactive 2D map of 2000 environmental sounds — click around to listen and see how similar sounds cluster together.

<!-- Tags: ["audio", "embeddings", "dimensionality-reduction", "pacmap"] -->

This notebook demonstrates:

- Registering a Hugging Face audio dataset as a 3LC Table using a custom `wav_audio` sample type.
- Computing audio embeddings with a pretrained AST model.
- Reducing the embeddings to 2D with PaCMAP for visual exploration in the Dashboard.

## Project Setup

In [ ]:
PROJECT_NAME = "3LC Tutorials - ESC-50 Audio Embeddings"
DATASET_NAME = "esc50"
TABLE_NAME = "initial"
MODEL_ID = "MIT/ast-finetuned-audioset-10-10-0.4593"
SAMPLE_RATE = 16_000  # AST expects 16 kHz
BATCH_SIZE = 16
DEVICE = None
INSTALL_DEPENDENCIES = True

In [ ]:
if INSTALL_DEPENDENCIES:
    %pip -q install "3lc[pacmap]" datasets transformers soundfile librosa
    %pip -q install -e .

## Imports

The `audio-sample-type` package (in this directory) registers a custom `wav_audio` sample type so 3LC Tables can store and play back WAV files.

In [ ]:
import librosa
import numpy as np
import tlc
import torch
from audio_sample_type import AudioWaveform, WavAudioSampleType
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import ASTFeatureExtractor, ASTModel

if DEVICE is None:
    if torch.cuda.is_available():
        device = "cuda:0"
    elif torch.backends.mps.is_available():
        device = "mps"
    else:
        device = "cpu"
else:
    device = DEVICE

device = torch.device(device)
print(f"Using device: {device}")

## Load ESC-50 from Hugging Face

The dataset contains 2000 five-second environmental audio recordings at 44.1 kHz.

In [ ]:
ds = load_dataset("ashraq/esc50", split="train")
print(f"Loaded {len(ds)} samples")
print(f"Columns: {ds.column_names}")
print(f"Example: category={ds[0]['category']}, target={ds[0]['target']}")

## Define category mapping

ESC-50 has 50 classes organized into 5 major categories. The mapping is based on the target index ranges (0–9: Animals, 10–19: Nature, 20–29: Human, 30–39: Domestic, 40–49: Urban).

In [ ]:
MAJOR_CATEGORIES = {
    0: "Animals",
    1: "Natural Soundscapes",
    2: "Human Sounds",
    3: "Domestic Sounds",
    4: "Urban Noises",
}


def get_major_category(target: int) -> int:
    """Map a class target (0-49) to its major category index (0-4)."""
    return target // 10


# Build the complete class list from the dataset
class_names: dict[int, str] = {}
for row in ds:
    target = row["target"]
    if target not in class_names:
        class_names[target] = row["category"]

class_names = dict(sorted(class_names.items()))
print(f"{len(class_names)} classes across {len(MAJOR_CATEGORIES)} categories")
for cat_idx, cat_name in MAJOR_CATEGORIES.items():
    classes_in_cat = [class_names[t] for t in range(cat_idx * 10, cat_idx * 10 + 10)]
    print(f"  {cat_name}: {', '.join(classes_in_cat)}")

## Create the 3LC Table

We write each audio sample as a WAV file using `WavAudioSampleType.schema()`, and include the class label, human-readable class name, and the major category for coloring the embedding plot.

In [ ]:
writer = tlc.TableWriter(
    project_name=PROJECT_NAME,
    dataset_name=DATASET_NAME,
    table_name=TABLE_NAME,
    schema={
        "audio": WavAudioSampleType.schema(display_name="Audio"),
        "label": tlc.schemas.CategoricalLabelSchema(classes=[class_names[i] for i in range(50)]),
        "category": tlc.schemas.CategoricalLabelSchema(classes=[MAJOR_CATEGORIES[i] for i in range(5)]),
        "class_name": tlc.schemas.StringSchema(),
    },
    if_exists="overwrite",
)

for row in tqdm(ds, desc="Writing audio table"):
    # Resample from 44.1 kHz to 16 kHz for the AST model
    waveform = row["audio"]["array"].astype(np.float32)
    sr = row["audio"]["sampling_rate"]
    if sr != SAMPLE_RATE:
        waveform = librosa.resample(waveform, orig_sr=sr, target_sr=SAMPLE_RATE)

    target = row["target"]
    writer.add_row(
        {
            "audio": AudioWaveform(waveform=waveform, sample_rate=SAMPLE_RATE),
            "label": target,
            "category": get_major_category(target),
            "class_name": row["category"],
        }
    )

table = writer.finalize()
print(f"Created table with {len(table)} rows")
print(f"Table URL: {table.url}")

## Extract audio embeddings with AST

We use the pretrained [Audio Spectrogram Transformer](https://huggingface.co/MIT/ast-finetuned-audioset-10-10-0.4593) to extract 768-dimensional embeddings. The model was trained on AudioSet, so it understands a wide range of environmental sounds. We use the `pooler_output` (average of CLS + distillation token) as the embedding for each clip.

In [ ]:
feature_extractor = ASTFeatureExtractor.from_pretrained(MODEL_ID)
model = ASTModel.from_pretrained(MODEL_ID).to(device)
model.eval()

print(f"Loaded AST model ({sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters)")
print(f"Embedding dimension: {model.config.hidden_size}")

In [ ]:
all_embeddings: list[list[float]] = []
waveforms_batch: list[np.ndarray] = []

for i in tqdm(range(len(table)), desc="Extracting embeddings"):
    sample = table[i]
    waveforms_batch.append(sample["audio"].waveform)

    if len(waveforms_batch) == BATCH_SIZE or i == len(table) - 1:
        inputs = feature_extractor(
            waveforms_batch,
            sampling_rate=SAMPLE_RATE,
            padding="max_length",
            return_tensors="pt",
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        embeddings = outputs.pooler_output.cpu().numpy()
        all_embeddings.extend(embeddings.tolist())
        waveforms_batch = []

print(f"Extracted {len(all_embeddings)} embeddings of dimension {len(all_embeddings[0])}")

## Add embeddings to the Table

We create a derived Table that carries forward the original rows and adds an `embedding` column.

In [ ]:
embedding_size = len(all_embeddings[0])

embedding_schema = tlc.schemas.Float32Schema(
    shape=(embedding_size,),
    number_role="nn_embedding",
    sample_type="hidden",
    writable=False,
)

schemas = {"embedding": embedding_schema, **table.rows_schema.values}

emb_writer = tlc.TableWriter(
    project_name=PROJECT_NAME,
    dataset_name=DATASET_NAME,
    table_name="with_embeddings",
    description="ESC-50 with AST embeddings",
    schema=schemas,
    if_exists="overwrite",
    input_tables=[table.url],
)

for i, row in tqdm(enumerate(table.table_rows), desc="Writing embeddings", total=len(table)):
    emb_writer.add_row({**row, "embedding": all_embeddings[i]})

table_with_embeddings = emb_writer.finalize()
print(f"Table with embeddings: {table_with_embeddings.url}")

## Reduce embeddings to 2D with PaCMAP

PaCMAP (Pairwise Controlled Manifold Approximation) preserves both local and global structure, making it well-suited for exploring clusters.

In [ ]:
reduced_table = tlc.reduction.reduce_embeddings(
    table_with_embeddings,
    method="pacmap",
    n_components=2,
    n_neighbors=15,
    retain_source_embedding_column=False,
)

print(f"Reduced table: {reduced_table.url}")
print()
print("Done! Open the table in the 3LC Dashboard to explore the embedding space.")
print("Color by 'Category' to see how the 5 major sound groups cluster.")
print("Click on points to listen to the audio samples.")